In [1]:
import pandas as pd
import json
from openai import OpenAI
import os

In [2]:
api_key = ""
with open("api kety.txt","r") as f:
    api_key = f.read()

In [3]:
client = OpenAI(
    api_key=api_key
)

In [4]:
kw_pediatrics = [
    '宝宝', '孩子', '小孩', '婴儿', '儿', '童', '幼', '小儿', '儿童', '幼儿', 
    '婴幼儿', '少儿', '新生儿', '男宝', '女宝', '男孩', '女孩', '儿科', '儿保', 
    '发育', '黄疸', '扁桃体', '奶', '母乳', '奶粉', '辅食', '早产', '退烧', 
    '积食', '出牙', '断奶', '长高', '多动症', '抽动症', '满月', '周岁', '百天', 
    '打疫苗', '接种', '厌食', '挑食', '尿布', '纸尿裤', '早教', '小儿科', '生长',
    '宝宝', '孩子', '小孩', '婴儿', '儿', '童', '幼', '小儿', '儿童', '幼儿', 
    '婴幼儿', '少儿', '新生儿', '男宝', '女宝', '男孩', '女孩', '儿科', '儿保', 
    '发育', '黄疸', '扁桃体', '奶', '母乳', '奶粉', '辅食', '早产', '退烧', 
    '积食', '出牙', '断奶', '长高', '多动症', '抽动症', '满月', '周岁', '百天', 
    '打疫苗', '接种', '厌食', '挑食', '尿布', '纸尿裤', '早教', '小儿科', '生长'
]

kw_internal = [
    '胃', '肠', '心', '肝', '肺', '肾', '血压', '血糖', '呼吸', '消化', 
    '内分泌', '内科', '心血管', '神经', '甲状腺', '糖尿病', '高血压', '高血脂', 
    '心脏病', '冠心病', '心梗', '脑梗', '癫痫', '慢性病', '胃炎', '溃疡', 
    '腹泻', '便秘', '脂肪肝', '乙肝', '硬化', '胆结石', '肾炎', '尿毒症', 
    '痛风', '风湿', '关节炎', '贫血', '头痛', '头晕', '失眠', '心悸', '胸闷', 
    '哮喘', '支气管', '感冒', '病毒', '发热', '血管', '代谢',
    '胃', '肠', '心', '肝', '肺', '肾', '血压', '血糖', '呼吸', '消化', 
    '内分泌', '内科', '心血管', '神经', '甲状腺', '糖尿病', '高血压', '高血脂', 
    '心脏病', '冠心病', '心梗', '脑梗', '癫痫', '慢性病', '胃炎', '溃疡', 
    '腹泻', '便秘', '脂肪肝', '乙肝', '硬化', '胆结石', '肾炎', '尿毒症', 
    '痛风', '风湿', '关节炎', '贫血', '头痛', '头晕', '失眠', '心悸', '胸闷', 
    '哮喘', '支气管', '感冒', '病毒', '发热', '血管', '代谢'

]

kw_oncology = [
    '肿瘤', '癌', '瘤', '化疗', '放疗', '转移', '恶性', '靶向', '肺癌', '肠癌', 
    '直肠癌', '胃癌', '肝癌', '胰腺癌', '乳腺癌', '宫颈癌', '卵巢癌', '食道癌', 
    '前列腺癌', '甲状腺癌', '白血病', '淋巴瘤', '黑色素瘤', '肉瘤', '腺癌', 
    '血管瘤', '癌细胞', '晚期', '早期', '中期', '良性', '息肉', '囊肿', '结节', 
    '穿刺', '活检', '复发', '生存期', '预后', '基因检测', '标志物', '抗癌', 
    '免疫治疗', '副作用', '姑息', '放化疗', '切除', '微创',
    '肿瘤', '癌', '瘤', '化疗', '放疗', '转移', '恶性', '靶向', '肺癌', '肠癌', 
    '直肠癌', '胃癌', '肝癌', '胰腺癌', '乳腺癌', '宫颈癌', '卵巢癌', '食道癌', 
    '前列腺癌', '甲状腺癌', '白血病', '淋巴瘤', '黑色素瘤', '肉瘤', '腺癌', 
    '血管瘤', '癌细胞', '晚期', '早期', '中期', '良性', '息肉', '囊肿', '结节', 
    '穿刺', '活检', '复发', '生存期', '预后', '基因检测', '标志物', '抗癌', 
    '免疫治疗', '副作用', '姑息', '放化疗', '切除', '微创'
]

In [5]:
def clean_basic_data(filepath):
    try:
        df = pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding='gbk')
        
    if 'ask' in df.columns and 'answer' in df.columns:
        df = df[['department', 'title', 'ask', 'answer']].rename(columns={'ask': 'question', 'answer': 'human_answer'})
    elif 'question' in df.columns and 'human_answer' in df.columns:
        df = df[['department', 'title', 'question', 'human_answer']]
    
    df = df.dropna(subset=['question', 'human_answer'])
    
    # NEW: Dynamically drop ANY row where ANY column is exactly "无"
    for col in df.columns:
        df = df[df[col].astype(str).str.strip() != "无"]
        
    return df

In [6]:
def keyword_filter_pool(df, domain_keywords):
    def has_keyword(row):
        text = str(row['title']) + " " + str(row['question'])
        return any(kw in text for kw in domain_keywords)
        
    mask = df.apply(has_keyword, axis=1)
    return df[mask].copy()

In [7]:
def evaluate_row_with_llm(question_text, domain):
    system_prompt = f"""You are a strict medical dataset curator for the '{domain}' domain.
Analyze the provided Chinese medical question. Return a JSON object.

CRITICAL RULES:
1. PEDIATRICS OVERRIDE (ACCEPT CHILD + IM/SURGERY): 
   - If the patient is a CHILD (baby, infant, son/daughter, toddler, age < 18) AND the current domain is 'Pediatrics', you MUST mark 'is_valid: true'. Do NOT reject it.
   - If the patient is a CHILD AND the current domain is 'Internal Medicine' or 'Oncology', mark 'is_valid: false' (Reason: Pediatric patient. Belongs to Pediatrics).
   - For the 'Pediatrics' domain, ACCEPT questions involving Internal Medicine conditions (e.g., viral infections, fevers) or general pediatric surgery (e.g., hernias).

2. THE ONCOLOGY EXCEPTION (REJECT CHILD + CANCER): 
   - If a query involves a child with CANCER, TUMORS, or LEUKEMIA, REJECT IT (is_valid: false) across ALL domains, including Pediatrics.

3. FETAL VS. NEONATAL (PREGNANCY EXCLUSION):
   - Queries about a fetus *during* pregnancy (e.g., fetal ultrasounds, fetal renal dysplasia) are Obstetrics/Gynecology. REJECT them.
   - Queries about a baby *after birth* (neonatal) are Pediatrics. ACCEPT them for the 'Pediatrics' domain.

4. THIRD-PARTY QUERIES: Well-wishers asking about an adult (e.g., "My dad has cancer") are VALID for Adult domains (Oncology/IM). Well-wishers asking about a child are VALID for Pediatrics.

5. EXCLUSIONS: Reject purely emotional/therapeutic venting, administrative hospital queries, and generic lifestyle/diet questions lacking clinical pathology.

6. JSON FORMATTING (TOKEN SAVING):
   - If 'is_valid' is true, return ONLY: {{"is_valid": true}}. DO NOT include 'reason' or 'english_translation'.
   - If 'is_valid' is false, you MUST include 'reason' AND 'english_translation'. Example: {{"is_valid": false, "reason": "...", "english_translation": "..."}}

EXAMPLES:
1. Child with IM/Surgical condition: "医生，我六个月的宝宝得了疝气，或者得了手足口病，怎么办？"
If domain is Pediatrics: {{"is_valid": true}}
If domain is Internal Medicine: {{"is_valid": false, "reason": "Patient is a child. Belongs to Pediatrics.", "english_translation": "Doctor, my 6-month-old baby has a hernia, or has hand-foot-and-mouth disease, what should I do?"}}

2. Child + Cancer (Reject everywhere): "医生，我五岁的儿子查出白血病，怎么办？"
{{"is_valid": false, "reason": "Ambiguous cross-domain: Pediatric patient with an Oncology condition.", "english_translation": "Doctor, my five-year-old son was diagnosed with leukemia, what should I do?"}}

3. Fetal/Pregnancy (Reject everywhere): "怀孕20周，四维彩超发现胎儿肾发育不良怎么办？"
{{"is_valid": false, "reason": "Obstetrics query about a fetus during pregnancy.", "english_translation": "I am 20 weeks pregnant, and the 4D ultrasound shows fetal renal dysplasia, what should I do?"}}

4. Adult Well-wisher (Valid IM): "我老公最近总是咳嗽，发烧，是乙肝引起的吗？"
If domain is Internal Medicine: {{"is_valid": true}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_text}
        ],
        temperature=0.0
    )
    
    return json.loads(response.choices[0].message.content)

In [8]:
def generate_clean_dataset(input_path, output_path, domain_name, domain_keywords, target_count=1000):
    print(f"\n--- Starting pipeline for {domain_name} ---")
    df_raw = clean_basic_data(input_path)
    df_pool = keyword_filter_pool(df_raw, domain_keywords)
    
    print(f"Keyword pool size for {domain_name}: {len(df_pool)} rows.")
    
    valid_rows = []
    evaluated_indices = set()
    
    while len(valid_rows) < target_count:
        needed = target_count - len(valid_rows)
        
        # Exclude ALL previously evaluated rows (accepted OR rejected)
        remaining_df = df_pool.loc[~df_pool.index.isin(evaluated_indices)]
        
        if len(remaining_df) == 0:
            print(f"WARNING: Exhausted all keyword-filtered rows. Only achieved {len(valid_rows)}/{target_count}.")
            break
            
        # Sample slightly more than needed to prevent the loop from slowing down due to rejections
        sample_size = min(needed * 2, len(remaining_df))
        
        # Dynamic random state ensures reproducible but unique samples each iteration
        current_sample = remaining_df.sample(n=sample_size, random_state=len(evaluated_indices))
        
        for idx, row in current_sample.iterrows():
            evaluated_indices.add(idx)  # Track the index so it's NEVER sampled again
            question_text = str(row['title']) + " " + str(row['question'])
            
            evaluation = evaluate_row_with_llm(question_text, domain_name)
            
            if evaluation.get('is_valid') is True:
                row_data = row.to_dict()
                # We do NOT save english_translation here to save API tokens
                valid_rows.append(row_data)
                
                if len(valid_rows) % 100 == 0:
                    print(f"[{domain_name}] Progress: {len(valid_rows)}/{target_count} clean rows extracted.")
                    
                if len(valid_rows) == target_count:
                    break
            else:
                # We ONLY print translations and reasons for rejected rows
                translation = evaluation.get('english_translation', 'No translation provided')
                reason = evaluation.get('reason', 'No specific reason given')
                print(f"-> REJECTED [{domain_name}]: {translation} | Reason: {reason}")

    final_df = pd.DataFrame(valid_rows)
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Successfully saved {len(final_df)} highly pure, non-redundant rows to {output_path}\n")
    return final_df

In [9]:
if __name__ == "__main__":
    #generate_clean_dataset('Orginal Datasets/Pediatrics.csv', 'Pediatrics 1000 Human.csv', 'Pediatrics', kw_pediatrics)
    generate_clean_dataset('Orginal Datasets/Internal Medicine.csv', 'IM 1000 Human.csv', 'Internal Medicine', kw_internal)
    #generate_clean_dataset('Orginal Datasets/Oncology.csv', 'Oncology 1000 Human.csv', 'Oncology', kw_oncology)


--- Starting pipeline for Internal Medicine ---
Keyword pool size for Internal Medicine: 151875 rows.
-> REJECTED [Internal Medicine]: I often can't sleep at night, is it insomnia? Basic information: Male, 18 years old. Condition description: I am a senior high school student, often can't sleep at night, I don't know if it's insomnia, and during the day I feel dizzy and have a headache. I want to ask if there are any good health care methods, thank you. | Reason: Patient is a child. Belongs to Pediatrics.
-> REJECTED [Internal Medicine]: What are some folk remedies to completely cure bronchitis? | Reason: Query lacks clinical pathology and is focused on alternative remedies.
-> REJECTED [Internal Medicine]: How can college students prevent acute gastroenteritis in daily life? In winter, how can college students prevent the recurrence of acute gastroenteritis during survival practice? | Reason: Generic lifestyle question lacking clinical pathology.
-> REJECTED [Internal Medicine]: Seve

In [ ]:
# # def repair_and_topup_dataset(current_csv, raw_csv, domain_name, domain_keywords, target_count=1000):
#     print(f"\n--- Repairing {domain_name} ---")
    
#     # 1. Load the existing 1000 rows
#     try:
#         df_current = pd.read_csv(current_csv, encoding='utf-8-sig')
#     except FileNotFoundError:
#         print(f"File {current_csv} not found. Skipping repair.")
#         return None
        
#     initial_len = len(df_current)
    
#     # 2. Identify and drop rows with "无" in ANY column
#     for col in df_current.columns:
#         df_current = df_current[df_current[col].astype(str).str.strip() != "无"]
#     df_current = df_current.dropna(subset=['question', 'human_answer'])
    
#     kept_rows = len(df_current)
#     needed = target_count - kept_rows
#     print(f"[{domain_name}] Dropped {initial_len - kept_rows} invalid rows (contained '无' or NaNs).")
    
#     if needed <= 0:
#         print(f"[{domain_name}] Dataset is perfectly clean. No top-up needed.")
#         df_current.to_csv(current_csv, index=False, encoding='utf-8-sig')
#         return df_current
        
#     print(f"[{domain_name}] Need {needed} new rows to reach {target_count}.")

#     # 3. Load the raw dataset to extract new rows
#     df_raw = clean_basic_data(raw_csv)
    
#     # Ensure the raw pool also drops any rows where ANY column has "无"
#     for col in df_raw.columns:
#         df_raw = df_raw[df_raw[col].astype(str).str.strip() != "无"]
        
#     df_pool = keyword_filter_pool(df_raw, domain_keywords)
    
#     # 4. Exclude questions we already have in df_current to prevent duplicates
#     existing_questions = set(df_current['question'].astype(str).tolist())
#     df_pool = df_pool[~df_pool['question'].astype(str).isin(existing_questions)]
    
#     valid_new_rows = []
#     evaluated_indices = set()
#     initial_sample_size = int(needed * 1.5)

#     while len(valid_new_rows) < needed:
#         remaining_df = df_pool.loc[~df_pool.index.isin(evaluated_indices)]
        
#         if len(remaining_df) == 0:
#             print("WARNING: Exhausted pool!")
#             break
            
#         num_to_sample = min(len(remaining_df), initial_sample_size)
#         current_sample = remaining_df.sample(n=num_to_sample, random_state=len(evaluated_indices))

#         for idx, row in current_sample.iterrows():
#             evaluated_indices.add(idx)
#             question_text = str(row['title']) + " " + str(row['question'])
            
#             evaluation = evaluate_row_with_llm(question_text, domain_name)
            
#             if evaluation.get('is_valid') is True:
#                 row_data = row.to_dict()
#                 row_data['english_translation'] = evaluation.get('english_translation')
#                 valid_new_rows.append(row_data)
                
#                 if len(valid_new_rows) == needed:
#                     break
#             else:
#                 print(f"-> REJECTED [{domain_name}]: {evaluation.get('reason')}")
                
#         initial_sample_size = (needed - len(valid_new_rows)) + 20

#     # 5. Combine the old clean rows with the new LLM-approved rows
#     df_new = pd.DataFrame(valid_new_rows)
#     df_final = pd.concat([df_current, df_new], ignore_index=True)
    
#     # Ensure exactly 1000 rows just in case
#     df_final = df_final.head(target_count) 
    
#     # Overwrite the original CSV with the pristine data
#     df_final.to_csv(current_csv, index=False, encoding='utf-8-sig')
#     print(f"Successfully repaired and saved {len(df_final)} pristine rows to {current_csv}")
    
#     return df_final

In [ ]:
# if __name__ == "__main__":
#     # 1. Repair Pediatrics
#     repair_and_topup_dataset('Pediatrics 1000 Human.csv', 'Orginal Datasets/Pediatrics.csv', 'Pediatrics', kw_pediatrics)
    
#     # 2. Repair Internal Medicine (Uncomment once you have generated its initial 1000 rows)
#     # repair_and_topup_dataset('IM 1000 Human.csv', 'Orginal Datasets/Internal Medicine.csv', 'Internal Medicine', kw_internal)
    
#     # 3. Repair Oncology (Uncomment once you have generated its initial 1000 rows)
#     # repair_and_topup_dataset('Oncology 1000 Human.csv', 'Orginal Datasets/Oncology.csv', 'Oncology', kw_oncology)

In [10]:
def generate_keyword_only_dataset(input_path, output_path, pos_keywords, neg_keywords_lists, target_count=1000):
    print(f"--- Starting Strict Keyword Pipeline for {output_path} ---")
    df_raw = clean_basic_data(input_path)
    
    def is_strictly_valid(row):
        text = str(row['title']) + " " + str(row['question'])
        has_pos = any(kw in text for kw in pos_keywords)
        has_neg = any(kw in text for neg_list in neg_keywords_lists for kw in neg_list)
        return has_pos and not has_neg
        
    mask = df_raw.apply(is_strictly_valid, axis=1)
    df_filtered = df_raw[mask].copy()
    
    if len(df_filtered) < target_count:
        print(f"WARNING: Only found {len(df_filtered)} valid rows, falling short of {target_count}.")
        df_final = df_filtered
    else:
        df_final = df_filtered.sample(n=target_count, random_state=42)
        
    # Create directory if it doesn't exist to prevent OSErrors
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        
    df_final.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Successfully saved {len(df_final)} pure rows to {output_path}\n")
    return df_final

In [12]:
if __name__ == "__main__":
    # 1. Pediatrics (Exclude IM and Oncology keywords)
    # generate_keyword_only_dataset(
    #     input_path='Orginal Datasets/Oncology.csv', 
    #     output_path='Oncology 1000 Human.csv', 
    #     pos_keywords=kw_oncology,
    #     neg_keywords_lists=[kw_internal, kw_pediatrics]
    # )
    
    # # 2. Internal Medicine (Exclude Pediatrics and Oncology keywords)
    # generate_keyword_only_dataset(
    #     input_path='C:/Users/Shravan Kumar N/Desktop/Mini Project/Datasets/Chinese Medical Q & A/Raw datasets/Internal Medicine.csv', 
    #     output_path='C:/Users/Shravan Kumar N/Desktop/Mini Project/Datasets/Chinese Medical Q & A/Preprocessed by us/IM 1000 Human.csv', 
    #     pos_keywords=kw_internal,
    #     neg_keywords_lists=[kw_pediatrics, kw_oncology]
    # )
    
    # 3. Oncology (Exclude Pediatrics and IM keywords)
    generate_keyword_only_dataset(
        input_path='Orginal Datasets/Oncology.csv', 
        output_path='Oncology 1000 Human.csv', 
        pos_keywords=kw_oncology,
        neg_keywords_lists=[kw_pediatrics, kw_internal]
    )

--- Starting Strict Keyword Pipeline for Oncology 1000 Human.csv ---
Successfully saved 1000 pure rows to Oncology 1000 Human.csv

